# NB_Comparatif : Comparaison algorithmes OCR

Ce notebook documente le comparatif des approches OCR évaluées sur le corpus factures Kaggle (batch_1).

**Metriques retenues :** CER (Character Error Rate), WER (Word Error Rate), taille modèle, latence inférence.

**Corpus :** factures Kaggle batch_1, meme split train/val que NB3 (factures, sans chevauchement).

Les resultats TrOCR proviennent de runs d'expérimentations menés en parallèle (`notebooks/NB_experiment_TrOCR.ipynb`).

Les resultats PaddleOCR REC proviennent de NB3 (`notebooks/NB3_Fine-tuning_DETRapidOCR_RECPaddleOCR.ipynb`).

## 1. TrOCR baseline (zero-shot)

Modele : `microsoft/trocr-base-printed` (pre-entraine SROIE, architecture Transformer ViT-12 + BERT-12).
Evaluation zero-shot sur 100 factures Kaggle (seed=42, crops via PaddleOCR DET).
Source : run consigné dans `NB_experiment_TrOCR.ipynb`, cell 41

In [1]:
# Output -- run NB_experiment_TrOCR cell 41
# Evaluation TrOCR pré-entraine sur 100 factures Kaggle batch_1
# Aucun rerun ici => valeurs issues du notebook archive.

baseline_metrics = {
    "CER": 0.8505,    # 85.05%
    "WER": 0.9668,    # 96.68%
    "params_M": 247,
    "latency_ms": 500.0,  # mesure indicative sur GPU L4
    "corpus": "100 factures, seed=42, crops PaddleOCR DET",
    "source": "NB_experiment_TrOCR, cell 41",
}

print(f"TrOCR baseline -- CER : {baseline_metrics['CER']:.2%}")
print(f"TrOCR baseline -- WER : {baseline_metrics['WER']:.2%}")
print(f"Taille modele : {baseline_metrics['params_M']}M parametres")
print(f"Latence indicative : ~{baseline_metrics['latency_ms']:.0f} ms/crop (GPU L4)")

TrOCR baseline -- CER : 85.05%
TrOCR baseline -- WER : 96.68%
Taille modele : 247M parametres
Latence indicative : ~500 ms/crop (GPU L4)


**Lecture :** Un CER de 85% sur factures correspond au comportement attendu d'un modèle pré-entraine sur SROIE.

Le domain gap est fort : polices, mises en page et qualité de scan des factures Kaggle different du corpus SROIE.

Ce résultat confirme la nécessité d'un fine-tuning sur le corpus cible.

## 2. TrOCR fine-tuné (Phase 2 : degel -4 couches ViT)

Même architecture que le baseline. Fine-tuning en deux phases :
- Phase 1 : décodeur seul entrainé (5 epochs, LR=1e-4, encoder gelé)
- Phase 2 : dégel des 4 dernières couches ViT (3 epochs, LR=1e-5)

Corpus d'entrainement : 17 001 paires (split par facture, overlap=0).

Source : run consigné dans `NB_experiment_TrOCR.ipynb`, cells 123-124

In [2]:
# Output -- run NB_experiment_TrOCR, cells 123-124
# Evaluation TrOCR fine-tune Phase 2 (generation_max_length=32, bugfix applique)
# Aucun rerun ici => valeurs issues du notebook archive.

phase1_metrics = {
    "CER": 0.1159,   # 11.59%
    "WER": 0.2249,   # 22.49%
    "epochs": 5,
    "phase": "decodeur seul (encoder gele)",
}

phase2_metrics = {
    "CER": 0.0903,   # 9.03%
    "WER": 0.1862,   # 18.62%
    "epochs": 3,
    "eval_loss": 0.3341,
    "phase": "degel -4 couches ViT",
    "source": "NB_experiment_TrOCR, cell 124 (apres bugfix generation_max_length=32)",
}

print("Phase 1 (decodeur seul) :")
print(f"  CER : {phase1_metrics['CER']:.2%} | WER : {phase1_metrics['WER']:.2%}")
print()
print("Phase 2 (degel -4 couches ViT) :")
print(f"  CER : {phase2_metrics['CER']:.2%} | WER : {phase2_metrics['WER']:.2%}")
print(f"  eval_loss : {phase2_metrics['eval_loss']}")
print(f"  Source    : {phase2_metrics['source']}")

Phase 1 (decodeur seul) :
  CER : 11.59% | WER : 22.49%

Phase 2 (degel -4 couches ViT) :
  CER : 9.03% | WER : 18.62%
  eval_loss : 0.3341
  Source    : NB_experiment_TrOCR archive, cell 124 (apres bugfix generation_max_length=32)


**Lecture :** Le fine-tuning réduit significativement le CER (85% → 9%) mais maintient un coût en paramètres et en latence identique au baseline (247M params, ~500ms/crop).

Le gain Phase 2 vs Phase 1 est limité (+2.5 pp CER) avec un budget GPU supplémentaire de ~25 min (3189 steps sur L4).

## 3. PaddleOCR REC fine-tuné (ce livrable)

Architecture CRNN : MobileNetV3 (backbone) + BiLSTM (sequence modeling) + CTC (decoder).

Fine-tuning REC seul ; la détection est assurée par RapidOCR (export ONNX du DBNet PP-OCR). L'équivalence fonctionnelle entre RapidOCR DET et PaddleOCR DET a été vérifiée dans `NB_DET_Benchmark.ipynb` (match rate 100 % à IoU >= 0.5, n=20).

Les métriques complètes (trajectoire, courbes, tests A/B) sont dans NB3 (`NB3_Fine-tuning_DETRapidOCR_RECPaddleOCR.ipynb`).

Rappel des valeurs clés ci-dessous pour le tableau comparatif.

In [ ]:
# Rappel metriques PaddleOCR REC -- source : NB3, best checkpoint epoch 34 + benchmark L4 NB3 §8.4

paddle_norm_edit_dis = 0.998111   # val interne batch_1, epoch 34 (best checkpoint retenu)
paddle_cer = 1 - paddle_norm_edit_dis

# Latence mesurée sur L4 (NB3 §8.4, 100 crops val, seed=42)
# Source : models/PaddleOCR_Invoice_v2/latency_benchmark.json
paddle_latency_batch1_ms = 16.4080   # worst case, 1 crop par appel (scénario FastAPI /ocr)
paddle_latency_batch12_ms = 3.3902  # régime NB3 §9, facture complète
paddle_fps_batch12 = 294.97
paddle_params_M = 8

print(f"CER proxy (1 - norm_edit_dis) : {paddle_cer:.4%}")
print(f"Latence (batch=1, worst case) : {paddle_latency_batch1_ms:.2f} ms/crop")
print(f"Latence (batch=12, pipeline)  : {paddle_latency_batch12_ms:.2f} ms/crop ({paddle_fps_batch12:.0f} fps)")
print(f"Taille modele                 : ~{paddle_params_M}M parametres")
print(f"Hardware                      : GPU: NVIDIA L4")
print(f"Source                        : NB3 §8.4 (latency_benchmark.json) + best checkpoint epoch 34")

## 4. Tableau comparatif synthetique

| Modèle | Architecture | CER | WER | Paramètres | Latence |
|---|---|---|---|---|---|
| TrOCR baseline | Transformer ViT-12+BERT-12 | 85.05% | 96.68% | 247M | ~500 ms/crop (L4) |
| TrOCR fine-tuné (Phase 2) | idem + adaptation domaine | 9.03% | 18.62% | 247M | ~500 ms/crop (L4) |
| **PaddleOCR REC fine-tuné** | CRNN MobileNetV3+BiLSTM+CTC | **0.19%** | - | **~8M** | **3.39 ms/crop** (batch=12) / 16.4 ms/crop (batch=1), L4 |

**Conditions :** corpus factures Kaggle batch_1, même type de corpus pour les trois, évaluation au niveau ligne (crop).

Le CER PaddleOCR est un proxy : `CER ≈ 1 - norm_edit_dis` (métrique native PaddleOCR REC, best checkpoint epoch 34, val_norm_edit_dis = 0.998111).

WER non disponible pour PaddleOCR REC : la métrique native est `norm_edit_dis` au niveau caractère.

## 5. Verdict

PaddleOCR REC fine-tuné est retenu comme modèle du livrable pour trois raisons convergentes :

1. **Qualité de transcription :** CER proxy de 0.19 %, soit ~47x inférieur au TrOCR fine-tuné (9.03 %) et
   ~448x inférieur au baseline (85.05 %). L'écart est suffisant pour un usage applicatif.

2. **Efficacité paramétrique :** 8M params contre 247M pour TrOCR, soit un rapport de ~31x.
   Un modèle plus petit est plus facile à déployer (mémoire, latence, coût).

3. **Latence (L4, NB3 §8.4) :** 3.39 ms/crop en batch=12 (régime NB3 §9, facture complète, 295 fps) et 16.4 ms/crop en batch=1 (worst case, 1 crop par appel type FastAPI `/ocr`). Contre ~500 ms/crop pour TrOCR sur le même GPU, soit un rapport de ~147x en batch et ~30x en worst case. Compatible avec un traitement batch ou temps quasi réel sans infrastructure lourde.

4. **Cohérence DET / REC :** le DET utilisé (RapidOCR, export ONNX du DBNet PP-OCR) produit les mêmes
   boîtes que PaddleOCR DET (benchmark dédié `NB_DET_Benchmark.ipynb` : match rate 100 % à IoU >= 0.5)
   mais avec un cadrage plus serré sur lequel le REC a été fine-tuné. La cohérence train/infer du REC
   est préservée sans introduire un second runtime Paddle côté détection.

### Limites du comparatif

Les métriques TrOCR et PaddleOCR ne sont pas évaluées dans des conditions strictement identiques. Quatre sources de divergence sont à considérer :

- **Dénominateurs métriques différents** : TrOCR est évalué via HuggingFace `evaluate` (CER/WER standards, dénominateur = longueur de la référence). PaddleOCR REC expose `norm_edit_dis` natif, dont on déduit `cer_proxy = 1 - norm_edit_dis`. Les deux quantités mesurent la même famille d'erreurs mais ne sont pas strictement égales (dénominateur `max(|ref|, |pred|)` côté PaddleOCR vs `|ref|` côté HuggingFace).
- **Environnements matériels hétérogènes** : les runs TrOCR et PaddleOCR REC ont été exécutés sur infrastructures différentes selon les phases (Lightning L4 vs autres). Cela n'affecte pas les métriques finales de qualité mais empêche une comparaison temps/throughput strictement équitable.
- **Hyperparamètres adaptés à chaque architecture** : batch size, nombre d'epochs et learning rate schedules ont été choisis en fonction des contraintes propres à chaque modèle (247M params vs 8M params), volontairement non uniformisés pour laisser chaque architecture s'exprimer dans son régime.
- **Corpus de crops asymétrique** : les métriques TrOCR sont calculées sur des crops générés par PaddleOCR DET (documenté dans `NB_experiment_TrOCR.ipynb` §0.1), tandis que celles de PaddleOCR REC le sont sur des crops RapidOCR DET (cohérent avec son pipeline d'entraînement). Le benchmark dédié (`NB_DET_Benchmark.ipynb`) montre que les deux détecteurs produisent les mêmes boîtes au `unclip_ratio` près, donc l'asymétrie n'introduit pas de biais qualitatif significatif, mais elle est à mentionner pour transparence méthodologique.

La seed (fixée à 42 par convention sur l'ensemble des runs du projet) n'est **pas** une source de divergence.

La supériorité de PaddleOCR REC reste cependant suffisamment marquée (CER ~47x inférieur, latence ~147x inférieure en batch et ~30x en worst case sur L4, taille ~31x moindre) pour être robuste à ces différences méthodologiques.